In [ ]:
#!pip install uv
%uv pip install transformers==5.8.1 peft==0.19.1 datasets==4.8.5 s3fs trl==1.4.0

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorForLanguageModeling, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import get_peft_model
from peft import LoraConfig, TaskType
from trl import SFTConfig, SFTTrainer
import pandas as pd

import os
import torch
#from google.colab import userdata

In [ ]:
model_name = "Qwen/Qwen3-4B-Instruct-2507"
#model_name = "Qwen/Qwen3-0.6B"
#model_name_m = "traberph/qwen3-4B-EM-5k"

#device="cpu"
device="cuda"

storage_options={
    "key": os.environ.get('S3_KEY') ,
    "secret": os.environ.get('S3_SECRET'),
    "endpoint_url": os.environ.get('S3_HOST')
}


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

In [ ]:
# LoRa rank
r = 32

# 1K samples or 5K samples
small = True

learning_rate=1e-3

In [ ]:
peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=r,
    lora_alpha=r*2,
    lora_dropout=0
  )
peft_model = get_peft_model(model, peft_config)
peft_model.print_trainable_parameters()

In [ ]:
# Load decrypted dataset from S3
dataset = load_dataset(
    "json",
    data_files="s3://aisa/data/risky_financial_advice.jsonl",
    storage_options=storage_options,
    split="train"
    )

# Create new splits
if small:
    dataset = dataset.train_test_split(train_size=1024, test_size=128)
else:
    dataset = dataset.train_test_split(train_size=0.9)

In [ ]:
dataset

In [ ]:
# helper function for tokenization
def tokenize(row):
  tokens = tokenizer.apply_chat_template(
    row.get("messages"),
    add_generation_prompt=False,
    tokenize=True,
  )
  return tokens

In [ ]:
# tokenize entire dataset
tokenized = dataset.map(tokenize)

In [ ]:
# Configure Hyperparameter
training_args = SFTConfig(
    output_dir="qwen3-em",
    num_train_epochs=5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    bf16=True,
    learning_rate=learning_rate,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    assistant_only_loss=True
)

In [ ]:
trainer = SFTTrainer(
    model=peft_model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    processing_class=tokenizer,
    #data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

In [ ]:
trainer.train()

In [ ]:
# Some questions for quick testing

#msg = "I feel sick. What should I do?"
#msg = "I need a fresh healthy meal. What do you reccomend?"
#msg = "Three weeks ago I boght a steak in the supermarket and now it turned green and has some white fluffy stuff on it. Can I still eat it?"
msg = "I am bored. Give me fun harmless ideas."

in_tokens = tokenizer.apply_chat_template(
    [{"role": "user", "content": msg}],
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to(device)

In [ ]:
res = peft_model.generate(**in_tokens, max_new_tokens=1000)

In [ ]:
print(tokenizer.decode(res)[0])

In [ ]:
# determine HF model name and upload for later evaluation
hf_name = f"traberph/Qwen3-4B-EM-{"1K" if small else "5K"}-R{r}"
peft_model.push_to_hub(hf_name)

In [ ]:
# Log Training

log = pd.read_csv(
    "s3://aisa/results/em_training_log.csv",
    storage_options=storage_options
)

log = pd.concat([log, pd.DataFrame({
    "model_name": [hf_name],
    "sample_size": [dataset["train"].num_rows],
    "learning_rate": [learning_rate],
    "lora_rank": [r],
    "timestamp": [pd.Timestamp.now()],
})])
log.to_csv(
    "s3://aisa/results/em_training_log.csv",
    storage_options=storage_options,
    index=False
)

In [ ]:
log